# Masked persona-to-prompt retrieval: reference solution
Every word of a title (and its other forms) is masked in its prompt, so lexical matching fails
and the match has to be found from meaning. Plan: (1) embed titles and prompts with pre-trained
word vectors, (2) compute several similarity signals per (title, prompt) pair, (3) choose how
to combine them on held-out authors, because every test prompt comes from an author who has no
prompt in train, (4) rank the test prompts for every test title.

In [ ]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings("ignore")
SEED, K = 2026, 10
DATA, OUT = Path("./dataset/public"), Path("./working")
OUT.mkdir(parents=True, exist_ok=True)
train_q = pd.read_csv(DATA / "train.csv")
test_q = pd.read_csv(DATA / "test.csv")
train_p = pd.read_csv(DATA / "train_prompts.csv")
test_p = pd.read_csv(DATA / "test_prompts.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
print(train_q.shape, test_q.shape, train_p.shape, test_p.shape)
print("share of test prompts with masked words:", test_p.prompt.str.contains(r"\[MASK\]").mean().round(3))

# Align train titles with their own prompt so row i of each table is a true pair.
train_p = train_p.set_index("prompt_id").loc[train_q.prompt_ids].reset_index()

## Pre-trained word vectors
spaCy's `en_core_web_md` ships 300-d word vectors. If it is not installed, the notebook falls
back to a character n-gram TF-IDF space so it still runs end to end (with a lower score).

In [ ]:
try:
    import spacy
    nlp = spacy.load("en_core_web_md", disable=["parser", "ner", "tagger", "lemmatizer", "attribute_ruler", "senter"])
    VECTORS = True
except Exception as exc:  # model not available in this environment
    print("spaCy vectors unavailable, using TF-IDF fallback:", exc)
    VECTORS = False

MAX_CHARS, MAX_TOKENS = 3000, 300
clean = lambda texts: [t.replace("[MASK]", " ") for t in texts]

def token_vectors(texts):
    """Unit-normalised vectors of the content words of each text (stop words dropped)."""
    out = []
    for doc in nlp.pipe([t[:MAX_CHARS] for t in clean(texts)], batch_size=64):
        vs = [tok.vector for tok in doc if tok.has_vector and tok.is_alpha and not tok.is_stop]
        vs = np.array(vs[:MAX_TOKENS]) if vs else np.zeros((1, 300), dtype=np.float32)
        out.append(vs / (np.linalg.norm(vs, axis=1, keepdims=True) + 1e-9))
    return out

def mean_unit(vs_list):
    m = np.array([v.mean(0) for v in vs_list])
    return m / (np.linalg.norm(m, axis=1, keepdims=True) + 1e-9)

## Pairwise similarity signals
For a block of titles × prompts:
* `mean_cos`: cosine of the averaged title and prompt vectors,
* `soft_match`: for each title word, its best-matching prompt word; averaged over the title.
  This captures related words that survive masking (terminal -> shell, command, output),
* `head_cos`: cosine with the first 400 characters of the prompt, where the role is usually set,
* `char_cos`: residual character n-gram overlap,
* `mask_rate` and title length: masked prompts and long titles behave differently.

In [ ]:
def pair_features(titles, prompts):
    n_q, n_p = len(titles), len(prompts)
    feats = {}
    if VECTORS:
        tv, pv = token_vectors(titles), token_vectors(prompts)
        hv = token_vectors([p[:400] for p in prompts])
        feats["mean_cos"] = mean_unit(tv) @ mean_unit(pv).T
        feats["head_cos"] = mean_unit(tv) @ mean_unit(hv).T
        words = np.vstack(tv); owner = np.repeat(np.arange(n_q), [len(v) for v in tv])
        soft = np.zeros((n_q, n_p))
        for j, P in enumerate(pv):
            best = (P @ words.T).max(axis=0)                       # best prompt word per title word
            soft[:, j] = np.bincount(owner, weights=best, minlength=n_q) / np.bincount(owner, minlength=n_q)
        feats["soft_match"] = soft
    v = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True).fit(clean(prompts) + list(titles))
    feats["char_cos"] = (v.transform(titles) @ v.transform(clean(prompts)).T).toarray()
    mask_rate = np.array([p.count("[MASK]") / max(1, len(p.split())) for p in prompts])
    title_len = np.array([len(t.split()) for t in titles], dtype=float)
    feats["mask_x_len"] = np.outer(np.log1p(title_len), np.log1p(100 * mask_rate))
    # Standardise each signal within a query so the ranker compares prompts for the same title.
    return {k: (f - f.mean(1, keepdims=True)) / (f.std(1, keepdims=True) + 1e-9) for k, f in feats.items()}

def mrr_at_k(score):
    """score[i, j]: title i vs prompt j, with the true prompt of title i at column i."""
    rank = (score > np.diag(score)[:, None]).sum(1) + 1
    return float(np.where(rank <= K, 1.0 / rank, 0.0).mean())

## Validation on held-out authors
GroupKFold by `author_id`: each fold is a retrieval task over prompts written by authors that
no other fold contains, like the test set. Each signal is scored alone, then a weighted sum of
the vector signals is chosen by grid search on the same folds.

In [ ]:
import itertools
titles, prompts = train_q.title.tolist(), train_p.prompt.tolist()
folds = [pair_features([titles[i] for i in v], [prompts[i] for i in v])
         for _, v in GroupKFold(n_splits=4).split(train_q, groups=train_p.author_id)]
names = sorted(folds[0])

def cv(weights):
    return float(np.mean([mrr_at_k(sum(weights.get(k, 0.0) * F[k] for k in names)) for F in folds]))

single = {k: cv({k: 1.0}) for k in names}
grid = [dict(zip(["mean_cos", "head_cos", "soft_match", "char_cos"], w))
        for w in itertools.product([1.0], [0, 0.25, 0.5, 1.0], [0, 0.25, 0.5, 1.0], [0, 0.1, 0.25])]
scores = [cv(w) for w in grid]
weights = grid[int(np.argmax(scores))]
summary = pd.Series({**{f"single signal: {k}": v for k, v in single.items()},
                     "weighted sum (grid search)": max(scores)}, name="author-held-out MRR@10").round(4)
print(summary.sort_values().to_string())
print("chosen weights:", weights)

## Final ranking
Score every test title against every test prompt with the chosen weights and keep the top 10.

In [ ]:
F_test = pair_features(test_q.title.tolist(), test_p.prompt.tolist())
score = sum(weights.get(k, 0.0) * F_test[k] for k in names)
top = np.argsort(-score, axis=1)[:, :K]
ids = test_p.prompt_id.to_numpy()
submission = pd.DataFrame({"query_id": test_q.query_id, "prompt_ids": [" ".join(ids[r]) for r in top]})
assert len(submission) == len(sample) and set(submission.query_id) == set(sample.query_id)
submission.to_csv(OUT / "submission.csv", index=False)
submission.head()